<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 7.OpenCV脸部和眼睛检测

@－－－－DICE－－－－<br>
@  文件名：7.OpenCV脸部和眼睛检测.ipynb <br>
@  版本：V2.0 <br>
@  说明：OpenCV脸部和眼睛检测<br>

### 调整机械臂到达Home 点，便于获取图像

In [7]:
import math
from arm5dof_uservo import Arm5DoFUServo
from config import *

### 机械臂创建

In [8]:
arm = Arm5DoFUServo()

### 到达Home点

In [9]:
# 回到Home 点
arm.home()

In [10]:
#到达拍摄位置
arm.uservo.set_servo_angle(0,88,interval=1000)
arm.uservo.set_servo_angle(3,-80,interval=1000)
arm.uservo.wait()

In [11]:
# 导入必要的库
import numpy as np
import cv2

In [12]:
# 载入HAAR模型
face_cascade = cv2.CascadeClassifier('./images/haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('./images/haarcascade_eye.xml')

In [13]:
# 线程函数操作库
import threading # 线程
import ctypes
import inspect

# 线程结束代码
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError("PyThreadState_SetAsyncExc failed")
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)


In [14]:
# 创建显示控件
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg


face_imge = widgets.Image(format='jpeg', width=480, height=320)
display(face_imge)

Image(value=b'', format='jpeg', height='320', width='480')

In [15]:
def Video_display():
    capture = cv2.VideoCapture(0)
    while True:
        ret, frame = capture.read()
        if not ret:
            return ret, frame
        
        cframe_data = cv2.flip(frame,1)
        gray = cv2.cvtColor(cframe_data, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        for (x,y,w,h) in faces:
            cv2.rectangle(cframe_data,(x,y),(x+w,y+h),(255,0,0),2)
            roi_gray = gray[y:y+h, x:x+w]
            roi_color = cframe_data[y:y+h, x:x+w]
            print(int(x+w/2), int(y+h/2))
            eyes = eye_cascade.detectMultiScale(roi_gray)
            for (ex,ey,ew,eh) in eyes:
                cv2.rectangle(roi_color,(ex,ey),(ex+ew,ey+eh),(0,255,0),2)
        face_imge.value = bgr8_to_jpeg(cframe_data)
    capture.release()

In [16]:
t = threading.Thread(target=Video_display)
t.setDaemon(True)
t.start()

458 158
455 162
458 153
456 158
463 155
458 156
453 158
463 158
457 159
448 158
253 136
325 132
334 140
340 142
343 145
351 151
372 146
388 146
392 147
400 145
400 146
399 149
396 150
399 153
397 148
397 150
400 152
399 149
395 148
396 150
400 148
401 146
402 149
401 152
399 151
406 151
404 149
404 149
406 154
404 153
404 153
404 153
403 155
406 154
404 155
406 154
404 151
406 152
406 148
404 154
394 151
381 145
356 135
345 134
336 136
334 139
330 141
325 144
330 141
330 144
332 147
332 139
332 143
332 143
332 143
331 142
335 140
333 141
334 141
344 138
349 129
362 127
363 129
360 173
371 129
380 148
363 199
405 171
394 162
389 157
385 154
381 152
381 152
376 150
382 147
376 148
369 144
362 145
362 141
354 138
347 134
343 129
340 201
413 163
390 157
382 151
360 205
362 148
349 152
343 155
339 160
349 153
344 155
337 149
341 149
340 154
346 155
343 155
340 159
345 165
344 163
349 167
349 160
347 161
346 161
344 162
346 161
349 162
346 165
345 159
340 165
336 160
340 171
339 170
340 159


In [17]:
# 结束线程
stop_thread(t)